# Notebook 37｜模型外部验证与真实案例论文证据链整合

本 Notebook 冻结论文级最终证据链，并严格区分三种作用：

1. **ChemVista 盲法外部测试**：负责定量评价训练未见化合物的泛化性能；
2. **葡萄牙污水 DDA 案例**：负责展示已知污染物恢复、真实基质预测和风险排序；
3. **孟加拉国地表水 DIA 案例**：负责展示未知特征筛查、优先级排序及分层结构注释。

本步骤不重新训练、不调整阈值、不修改身份结果，也不会把葡萄牙开发集重叠化合物并入主外部验证。

## 运行方法

1. 选择 **Python (env_ms)** 内核；
2. 点击 **Kernel → Restart Kernel and Run All Cells**；
3. 最后一格必须显示 `MANUSCRIPT_EVIDENCE_SYNTHESIS_FROZEN`；
4. 11 项 QC 必须全部为 `PASS`。

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

assert 'env_ms' in sys.executable.lower(), f'请切换到 Python (env_ms) 内核；当前解释器：{sys.executable}'
cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs/pbt_msms_v1').exists()), None)
assert ROOT is not None, f'无法从 {cwd} 定位项目根目录'
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pbt_msms.manuscript_evidence_synthesis import FINAL_STATUS, run

print('Project root:', ROOT)
print('Python:', sys.executable)
print('Expected final status:', FINAL_STATUS)

In [ ]:
manifest = run(ROOT)
OUT = ROOT / 'outputs/pbt_msms_v1/manuscript_synthesis_37_v1'
print(json.dumps({
    'status': manifest['status'],
    'qc_all_required_pass': manifest['qc_all_required_pass'],
    'upstream_status': manifest['upstream_status'],
}, ensure_ascii=False, indent=2))

## 1. 盲法外部验证：论文的主要性能证据

下表只使用 ChemVista `novel_all` 未见化合物队列。分类端点报告 PR-AUC/ROC-AUC，回归端点报告 MAE、Spearman 相关及辅助指标。

In [ ]:
external = pd.read_csv(OUT / '37_external_validation_main.csv')
display(external[['endpoint', 'selected_pipeline', 'compound_n', 'primary_metric', 'primary_value', 'primary_ci95_low', 'primary_ci95_high', 'supporting_metric', 'supporting_value', 'interpretation']])

## 2. 三层证据的职责与结论上限

外部性能、已知污染物真实案例和未知物筛查案例不能相互替代，也不能简单合并样本量。

In [ ]:
case_studies = pd.read_csv(OUT / '37_case_study_summary.csv')
display(case_studies)

## 3. 葡萄牙小样本真值检查

该表保留样本量和 `DESCRIPTIVE_UNDERPOWERED` 状态，用于防止在论文中误写成第二套充分外部验证。

In [ ]:
portugal_truth = pd.read_csv(OUT / '37_portugal_supportive_truth_audit.csv')
display(portugal_truth)

## 4. 孟加拉国候选证据分层

结构层级是内部证据分层，不是鉴定概率；所有 CSI:FingerID/SIRIUS 结构仍需标准品 RT 与 MS/MS 确认。

In [ ]:
tiers = pd.read_csv(OUT / '37_bangladesh_evidence_tiers.csv')
display(tiers)

## 5. 可写结论、禁用表述与局限性

写论文时应直接以这两张冻结表为边界。

In [ ]:
claims = pd.read_csv(OUT / '37_evidence_claim_matrix.csv')
limitations = pd.read_csv(OUT / '37_limitations_register.csv')
display(claims)
display(limitations)

In [ ]:
figure_path = OUT / 'figures/37_integrated_evidence_chain.png'
display(Image(filename=str(figure_path), width=1500))

## 6. 可直接修改的中英文结果草稿

In [ ]:
draft_path = OUT / '37_manuscript_results_draft.md'
display(Markdown(draft_path.read_text(encoding='utf-8')))

## 7. 最终冻结与 QC

In [ ]:
qc = pd.read_csv(OUT / '37_synthesis_qc_checks.csv')
summary = json.loads((OUT / '37_synthesis_summary.json').read_text(encoding='utf-8'))
display(qc)
assert qc['status'].eq('PASS').all(), qc.loc[~qc['status'].eq('PASS')]
assert manifest['status'] == FINAL_STATUS
assert manifest['qc_all_required_pass'] is True
assert summary['status'] == FINAL_STATUS
print('FINAL STATUS:', manifest['status'])
print('Overall conclusion:', summary['overall_conclusion'])
print('输出目录:', OUT)